In [6]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression


def train_ticket_classifier(csv_path, model_path="../outputs/ticket_model.joblib"):
    df = pd.read_csv(csv_path)

    X = df["ticket_text"]
    y = df["category"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    model = Pipeline([
        ("tfidf", TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2)
        )),
        ("classifier", LogisticRegression(
            max_iter=1000,
            class_weight="balanced"
        ))
    ])

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    print(classification_report(y_test, predictions))

    joblib.dump(model, model_path)

    return model


def predict_ticket(model, ticket_text):
    prediction = model.predict([ticket_text])[0]
    probabilities = model.predict_proba([ticket_text])[0]

    confidence = probabilities.max()

    return {
        "ticket_text": ticket_text,
        "predicted_category": prediction,
        "confidence": round(confidence, 2)
    }
if __name__ == "__main__":
    model = train_ticket_classifier("../data/student_tickets.csv")
#I cannot log into my account
#I cannot register for my course because the system shows an error
    sample_ticket = "I cannot log into my account"
    result = predict_ticket(model, sample_ticket)

    print(result)


                     precision    recall  f1-score   support

  academic_advising       0.60      0.60      0.60         5
         admissions       0.80      0.80      0.80         5
course_registration       0.57      0.80      0.67         5
      financial_aid       0.67      0.40      0.50         5
         graduation       1.00      0.60      0.75         5
  technical_support       0.62      0.83      0.71         6

           accuracy                           0.68        31
          macro avg       0.71      0.67      0.67        31
       weighted avg       0.71      0.68      0.67        31

{'ticket_text': 'I cannot log into my account', 'predicted_category': 'technical_support', 'confidence': 0.28}
